## Demo 1: Timesheets

### Import the needed modules

* os for file system access
* pandas to work with data frames

In PowerShell I mostly use arrays of PSCustomObjects to store data and not the .NET classes like DataTable because of too much metadata.

In Python I will use data frames because I think is is the natural way to work with data here.

### What demo data do we have?

In [7]:
import os

data_path = r"..\data\timesheets"

for file in os.listdir(data_path):
    print(file)

DepartmentA.xlsx
DepartmentB.xlsx
DepartmentC.xlsx
README.md
sample.json


### Let's open the Excel files and have a look at the data

In [8]:
# Commented out so that the files don't open when running the script. Uncomment to open the files.

#os.startfile(r"..\data\timesheets\DepartmentA.xlsx")
#os.startfile(r"..\data\timesheets\DepartmentB.xlsx")
#os.startfile(r"..\data\timesheets\DepartmentC.xlsx")

### Let's import the data using pandas

In [9]:
import pandas as pd

df = pd.read_excel(
    r"..\data\timesheets\DepartmentA.xlsx",
    sheet_name="PersonA",
    skiprows=2,
)

df["start"] = df["date"] + pd.to_timedelta(df["time_from"].astype(str))
df["end"]   = df["date"] + pd.to_timedelta(df["time_to"].astype(str))

df


,date,time_from,time_to,project,task,start,end
0,2025-01-02,08:00:00,12:15:00,ProjectA,Testing,2025-01-02 08:00:00,2025-01-02 12:15:00
1,2025-01-02,13:00:00,15:00:00,ProjectB,Meeting,2025-01-02 13:00:00,2025-01-02 15:00:00


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       2 non-null      datetime64[us]
 1   time_from  2 non-null      object        
 2   time_to    2 non-null      object        
 3   project    2 non-null      str           
 4   task       2 non-null      str           
 5   start      2 non-null      datetime64[us]
 6   end        2 non-null      datetime64[us]
dtypes: datetime64[us](3), object(2), str(2)
memory usage: 244.0+ bytes


### Build a readable output with only the needed data.

In [11]:
import pandas as pd

input_frame = pd.read_excel(
    r"..\data\timesheets\DepartmentA.xlsx",
    sheet_name="PersonA",
    skiprows=2,
)

output_frame = input_frame.assign(
    Department="DepartmentA",
    Person="PersonA",
    Start=input_frame["date"] + pd.to_timedelta(input_frame["time_from"].astype(str)),
    End=input_frame["date"] + pd.to_timedelta(input_frame["time_to"].astype(str)),
    Project=input_frame.get("project"),
    Task=input_frame.get("task"),
)[["Department", "Person", "Start", "End", "Project", "Task"]]

output_frame

,Department,Person,Start,End,Project,Task
0,DepartmentA,PersonA,2025-01-02 08:00:00,2025-01-02 12:15:00,ProjectA,Testing
1,DepartmentA,PersonA,2025-01-02 13:00:00,2025-01-02 15:00:00,ProjectB,Meeting


### Move the code to a function that can import all files in a directory.

In [12]:
from import_xls_timesheet import import_xls_timesheet

timesheets = import_xls_timesheet(r"..\data\timesheets")

timesheets

📄 File: DepartmentA.xlsx
   ↳ Sheet: PersonA
   ↳ Sheet: PersonB
📄 File: DepartmentB.xlsx
   ↳ Sheet: PersonC
   ↳ Sheet: PersonD
📄 File: DepartmentC.xlsx
   ↳ Sheet: PersonC
   ↳ Sheet: PersonD


,Department,Person,Start,End,Project,Task
0,DepartmentA,PersonA,2025-01-02 08:00:00,2025-01-02 12:15:00,ProjectA,Testing
1,DepartmentA,PersonA,2025-01-02 13:00:00,2025-01-02 15:00:00,ProjectB,Meeting
2,DepartmentA,PersonB,2025-01-02 08:00:00,2025-01-02 11:00:00,ProjectA,Code review
3,DepartmentA,PersonB,2025-01-02 13:15:00,2025-01-02 14:45:00,ProjectB,Meeting
4,DepartmentB,PersonC,2025-01-02 08:00:00,2025-01-02 12:00:00,ProjectA,Testing
...,...,...,...,...,...,...
89,DepartmentC,PersonD,2025-01-27 08:00:00,2025-01-27 16:00:00,ProjectB,Workshop
90,DepartmentC,PersonD,2025-01-28 08:00:00,2025-01-28 16:00:00,ProjectB,Development
91,DepartmentC,PersonD,2025-01-29 08:00:00,2025-01-29 16:00:00,ProjectB,Code review
92,DepartmentC,PersonD,2025-01-30 08:00:00,2025-01-30 16:00:00,ProjectB,Code review


### Bonus: Extend the data frame by calculation the hours worked.

In [13]:
# Calculate hours
timesheets["Hours"] = (
    timesheets["End"] - timesheets["Start"]
).dt.total_seconds() / 3600

timesheets

,Department,Person,Start,End,Project,Task,Hours
0,DepartmentA,PersonA,2025-01-02 08:00:00,2025-01-02 12:15:00,ProjectA,Testing,4.25
1,DepartmentA,PersonA,2025-01-02 13:00:00,2025-01-02 15:00:00,ProjectB,Meeting,2.00
2,DepartmentA,PersonB,2025-01-02 08:00:00,2025-01-02 11:00:00,ProjectA,Code review,3.00
3,DepartmentA,PersonB,2025-01-02 13:15:00,2025-01-02 14:45:00,ProjectB,Meeting,1.50
4,DepartmentB,PersonC,2025-01-02 08:00:00,2025-01-02 12:00:00,ProjectA,Testing,4.00
...,...,...,...,...,...,...,...
89,DepartmentC,PersonD,2025-01-27 08:00:00,2025-01-27 16:00:00,ProjectB,Workshop,8.00
90,DepartmentC,PersonD,2025-01-28 08:00:00,2025-01-28 16:00:00,ProjectB,Development,8.00
91,DepartmentC,PersonD,2025-01-29 08:00:00,2025-01-29 16:00:00,ProjectB,Code review,8.00
92,DepartmentC,PersonD,2025-01-30 08:00:00,2025-01-30 16:00:00,ProjectB,Code review,8.00


### Setup everything to be able to work with Microsoft SQL Server

In [14]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

from connect_sql_instance import connect_sql_instance
from invoke_sql_query import invoke_sql_query
from write_sql_table import write_sql_table

### Create the target table in the Microsoft SQL Server

In [15]:
connection = connect_sql_instance(
    instance="127.0.0.1",
    database="TimeSheets",
    username="TimeSheets",
    password="Passw0rd!"
)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using SQL authentication
[VERBOSE] Disabling connection pooling
[VERBOSE] Opening connection
[VERBOSE] Returning connection object


In [16]:
create_table_query = '''
CREATE TABLE dbo.Timesheet (
  Department VARCHAR(100),
  Person     VARCHAR(100),
  Start      DATETIME2,
  "End"      DATETIME2,
  Project    VARCHAR(100),
  Task       VARCHAR(1000),
  CONSTRAINT Timesheet_PK 
  PRIMARY KEY (
    Department,
    Person,
    Start
  )
)'''

invoke_sql_query(
    connection=connection,
    query=create_table_query
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1


In [17]:
invoke_sql_query(
    connection=connection,
    query='TRUNCATE TABLE dbo.Timesheet'
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1


TODO: Insert step by step code here.

In [18]:
# If we don't have the variable timesheets or if it's empty, we should not attempt to write to the database
if 'timesheets' not in locals() or timesheets.empty:
    raise SystemExit("No timesheet data to write to the database.")

# If we don't have the variable connection or if it's empty, we should not attempt to write to the database
if 'connection' not in locals() or connection is None:
    raise SystemExit("No database connection available.")

write_sql_table(
    connection=connection,
    table="dbo.Timesheet",
    data=timesheets,
)

Importing data into [dbo].[Timesheet]
Inserting 94 rows...
94/94 rows inserted (100.0%) - 1304 rows/sec
Bulk insert complete.


In [19]:
result = invoke_sql_query(
    connection=connection,
    query='SELECT * FROM dbo.Timesheet',
)

result

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 94 rows


,Department,Person,Start,End,Project,Task
0,DepartmentA,PersonA,2025-01-02 08:00:00,2025-01-02 12:15:00,ProjectA,Testing
1,DepartmentA,PersonA,2025-01-02 13:00:00,2025-01-02 15:00:00,ProjectB,Meeting
2,DepartmentA,PersonB,2025-01-02 08:00:00,2025-01-02 11:00:00,ProjectA,Code review
3,DepartmentA,PersonB,2025-01-02 13:15:00,2025-01-02 14:45:00,ProjectB,Meeting
4,DepartmentB,PersonC,2025-01-02 08:00:00,2025-01-02 12:00:00,ProjectA,Testing
...,...,...,...,...,...,...
89,DepartmentC,PersonD,2025-01-27 08:00:00,2025-01-27 16:00:00,ProjectB,Workshop
90,DepartmentC,PersonD,2025-01-28 08:00:00,2025-01-28 16:00:00,ProjectB,Development
91,DepartmentC,PersonD,2025-01-29 08:00:00,2025-01-29 16:00:00,ProjectB,Code review
92,DepartmentC,PersonD,2025-01-30 08:00:00,2025-01-30 16:00:00,ProjectB,Code review


### Cleanup

In [20]:
# Commented out to prevent accidental deletion of the table. Uncomment to drop the table.

# invoke_sql_query(connection=connection,query='DROP TABLE dbo.Timesheet')